# Why Pydantic Exists

### The dynamic typing problem

In [1]:
def register_user(name,email,age):
    birth_year = 2026 - age
    user_info = {
        "name": name,
        "email": email,
        "age": age,
        "birth_year": birth_year
    }
    return user_info


In [3]:
register_user("John Doe", "john.doe@example .com", "thirty")

TypeError: unsupported operand type(s) for -: 'int' and 'str'

##### The bug was never really in the calculation. The bug was that nothing checked the incoming data at the door.

## Type hints: documentation, not enforcement

## Optional and Literal types

In [6]:
## Optional: It is possible to leave out fields of the Optional type when building a model instance.
## Literal: typing.Literal is used to restrict a field's value to an exact set of predefined choices.

### BaseModel vs. dataclass vs. plain class

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel

# Option 1: dataclass — clean syntax, ZERO validation
@dataclass
class UserDataclass:
    name: str
    email: str
    age: int

user = UserDataclass(name="Mohit", email="mohit@example.com", age="not a number")
print(user.age)   # "not a number" — accepted with no complaint at all

# Option 2: BaseModel — actually inspects the data
class UserModel(BaseModel):
    name: str
    email: str
    age: int

UserModel(name="Mohit", email="mohit@example.com", age="not a number")
# ValidationError: Input should be a valid integer,
# unable to parse string as an integer



not a number


ValidationError: 1 validation error for UserModel
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [8]:
# But a numeric STRING is coerced safely:
user = UserModel(name="Mohit", email="mohit@example.com", age="30")
print(user.age, type(user.age))   # 30 <class 'int'>

30 <class 'int'>


## ValidationError 

In [10]:
from pydantic import BaseModel, ValidationError

class SignupForm(BaseModel):
    username: str
    email: str
    age: int
    newsletter_opt_in: bool = False   # has a default -> optional

# Two equivalent creation styles
user_a = SignupForm(**{"username": "aditi28", "email": "a@x.com", "age": 28})
user_b = SignupForm.model_validate({"username": "aditi28", "email": "a@x.com", "age": 28})

try:
    SignupForm(username="incomplete_user")   # missing email AND age
except ValidationError as e:
    print(e)
# 2 validation errors for SignupForm
# email
#   Field required [type=missing, ...]
# age
#   Field required [type=missing, ...]

2 validation errors for SignupForm
email
  Field required [type=missing, input_value={'username': 'incomplete_user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
age
  Field required [type=missing, input_value={'username': 'incomplete_user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


## Serialization basics: model_dump() and model_dump_json()

In [12]:
user = SignupForm(username="aditi28", email="aditi@example.com", age=28, newsletter_opt_in=True)

user.model_dump()
# {'username': 'aditi28', 'email': 'aditi@example.com', 'age': 28, 'newsletter_opt_in': True}

user.model_dump_json()
# '{"username":"aditi28","email":"aditi@example.com","age":28,"newsletter_opt_in":true}'

user.model_dump_json(indent=2)   # pretty-printed for humans/logs

'{\n  "username": "aditi28",\n  "email": "aditi@example.com",\n  "age": 28,\n  "newsletter_opt_in": true\n}'

# Field Constraints & Custom Validators

#### Pydantic's Field() function defines runtime validation rules and metadata constraints for data models in Python, commonly used with frameworks like FastAPI. Key constraints include numeric boundaries (gt, lt), string lengths (min_length, max_length), and regular expression patterns (pattern)

In [15]:
from typing import Annotated
from pydantic import BaseModel, Field

# Direct style
class JobApplicationV1(BaseModel):
    full_name: str = Field(min_length=2, max_length=100)
    years_experience: int = Field(ge=0, le=50)
    portfolio_url: str = Field(pattern=r"^https?://.*")

# Annotated style — same behavior, more composable
class JobApplicationV2(BaseModel):
    full_name: Annotated[str, Field(min_length=2, max_length=100)]
    years_experience: Annotated[int, Field(ge=0, le=50)]
    email: Annotated[
        str,
        Field(description="Applicant's contact email", examples=["rohan@example.com"]),
    ]

print(JobApplicationV1(full_name="Rohan", years_experience=5, portfolio_url="https://rohan.dev"))
print(JobApplicationV2(full_name="Rohan", years_experience=5, email="rohan@example.com"))



full_name='Rohan' years_experience=5 portfolio_url='https://rohan.dev'
full_name='Rohan' years_experience=5 email='rohan@example.com'


## Built-in special types

In [20]:
## There are more common validation which can use emailstr,httpurl,Anyurl,secretstr.
from pydantic import BaseModel, EmailStr, HttpUrl, SecretStr

class Applicant(BaseModel):
    name: str
    email: EmailStr        # rejects "not-an-email"
    website: HttpUrl        # must be http:// or https://

class UserAccount(BaseModel):
    username: str
    password: SecretStr

print(Applicant(name="Rohan", email="rohan@example.com", website="https://rohan.dev"))
account = UserAccount(username="rohan99", password="super-secret-123")
print(account)                              # password shows as **********


name='Rohan' email='rohan@example.com' website=HttpUrl('https://rohan.dev/')
username='rohan99' password=SecretStr('**********')


In [19]:
print(account.password.get_secret_value())  # the real value, on purpose

super-secret-123


# Field Validators and Model Validators

In [ ]:
#### field_validator → one field, its own rules. model_validator → the whole model, rules across fields. Together, they express essentially any business rule you can describe in plain English.

In [ ]:
### model validators are tools in data validation libraries like Pydantic Docs to check data. A field validator checks one specific piece of data, while a model validator checks the whole object or multiple pieces togethe

In [21]:
from pydantic import BaseModel, model_validator

class SignupForm(BaseModel):
    username: str
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def passwords_must_match(self):
        if self.password != self.confirm_password:
            raise ValueError("password and confirm_password do not match")
        return self

# A richer example — mutually exclusive preferences
class JobApplication(BaseModel):
    remote_preferred: bool
    willing_to_relocate: bool

    @model_validator(mode="after")
    def check_relocation_logic(self):
        if self.remote_preferred and self.willing_to_relocate:
            raise ValueError("Can't be both remote-only AND willing to relocate")
        return self

In [22]:
form= SignupForm(username="rohan99", password="super-secret-123", confirm_password="confirm-secret-123")

ValidationError: 1 validation error for SignupForm
  Value error, password and confirm_password do not match [type=value_error, input_value={'username': 'rohan99', '...': 'confirm-secret-123'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [23]:
form= SignupForm(username="rohan99", password="super-secret-123", confirm_password="super-secret-123")

## Computed Field
A computed field in Pydantic is created using the @computed_field decorator. It lets you include dynamic, read-only properties in your model's serialized output (model_dump() or model_dump_json()) and JSON Schema. Standard Python @property methods are normally left out of dumps, but @computed_field fixes this.

In [26]:
from pydantic import BaseModel, computed_field

class JobApplication(BaseModel):
    full_name: str
    years_experience: int

    @computed_field
    @property
    def experience_tier(self) -> str:
        if self.years_experience < 2:
            return "junior"
        elif self.years_experience < 7:
            return "mid"
        return "senior"

app = JobApplication(full_name="Aditi Sharma", years_experience=6)
print(app.experience_tier)      # "mid" — accessed like a normal attribute
print(app.model_dump())         # includes experience_tier automatically


mid
{'full_name': 'Aditi Sharma', 'years_experience': 6, 'experience_tier': 'mid'}


In [28]:
app.years_experience = 10       # models are mutable by default
print(app.experience_tier)      # "senior" — always fresh, never stale
print(app.model_dump())  

senior
{'full_name': 'Aditi Sharma', 'years_experience': 10, 'experience_tier': 'senior'}


## Serialization control: exclude, include, exclude_unset

##### exclude={"password"} removes specific fields from the output.
##### include={"username"} is the inverse — only those fields, nothing else.
##### exclude_unset=True is the single most useful flag for PATCH-style partial updates: it only serializes fields the caller explicitly provided, letting default values (like an unset bio field) disappear from the output rather than silently overwriting other data.

#### exclude_none=True drops any field currently set to None.

In [31]:
user = UserAccount(username="rohan99", email="rohan@example.com", password="hunter2")



In [32]:
user.model_dump(exclude={"password"})
# {'username': 'rohan99', 'email': 'rohan@example.com'}

{'username': 'rohan99'}

In [33]:
user.model_dump(include={"username"})
# {'username': 'rohan99'}

{'username': 'rohan99'}

In [34]:
# The PATCH-update use case:
user.model_dump(exclude_unset=True)
# Only fields the caller actually SET appear — defaults that were
# never explicitly provided are omitted entirely.

{'username': 'rohan99', 'password': SecretStr('**********')}

## Nested Models

In [35]:
## Real data is almost never flat. One model as another's field type, and validation cascades automatically.

## Models inside models, lists of models

In [ ]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    state: str
    pin_code: str

class Applicant(BaseModel):
    name: str
    email: str
    address: Address        # a whole model, used as a field type

# Parsing straight from a nested dictionary — the common real-world case
incoming = {
    "name": "Rohan Mehta",
    "email": "rohan@example.com",
    "address": {"city": "Pune", "state": "Maharashtra", "pin_code": "411001"},
}
applicant = Applicant.model_validate(incoming)
print(applicant.address.city)   # "Pune" — dot-chain access, fully typed

# Lists of nested models work the same way
class WorkExperience(BaseModel):
    company: str
    role: str
    years: int

class Application(BaseModel):
    applicant: Applicant
    work_history: list[WorkExperience]



Pune


In [39]:
print(applicant.address.state)   # "Maharashtra" — dot-chain access, fully typed
print(applicant.address.pin_code)   # "411001" — dot-chain access, fully typed

Maharashtra
411001


## Pydantic Settings

In [40]:
## the problem with os.getenv()

In [41]:
## os.getenv() gives you strings, not typed values, which forces you to do a lot of manual conversion and validation.

## BaseSettings, .env files, and SecretStr

In [ ]:
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")

    api_key: SecretStr
    max_connections: int = Field(default=100, ge=1, le=1000)
    debug: bool = False

settings = AppSettings()   # reads from .env / environment automatically
print(settings.max_connections, type(settings.max_connections))   # 200 <class 'int'>
print(settings.api_key)                          # ********** (masked)
print(settings.api_key.get_secret_value())       # the real value, on purpose

## Pydantic in Production: FastAPI

##### Watching Pydantic validate real HTTP requests explains why it's described as "everywhere" in the Python ecosystem.